# Identificação de compras judiciais

Este notebook realiza uma análise das descrições de compras públicas para identificar compras relacionadas a demandas judiciais.

Etapas da análise:

1. setup de dados e carregar o arquivo `inputs/objeto-compra.csv`;
2. inspecionar o conjunto de dados;
3. criar regras e funções;
4. aplicar a função definida;
5. filtrar os registros identificados para análise;
6. exportar resultado para diretório de saída.

In [13]:
#----------------------------------
# Importar bibliotecas necessárias
#----------------------------------
import re
import pandas as pd
import unicodedata
from nltk.corpus import stopwords

#----------------------------------
# DIRETORIO
#----------------------------------
base_dir = "C:/Users/gabri/OneDrive/Área de Trabalho/joao/TB/cesta-de-precos-pncp"

#----------------------------------
# INPUTS
#----------------------------------
DATASET_COMPRAS = "tasks/verifica-compras-judiciais/inputs/objeto-compra.csv"

#----------------------------------
# OUTPUTS
#----------------------------------
OUTPUT_COMPRAS_JUDICIAIS = "tasks/verifica-compras-judiciais/outputs/compras-somente-judiciais.csv"

#----------------------------------
# Carrega o dataset de compras
#----------------------------------
compras_df = pd.read_csv(f"{base_dir}/{DATASET_COMPRAS}")

##  Inspeção de `compras_df`
- O dataset em questão possui dados de várias compras, não somente medicamentos

In [14]:
print(f"Total de compras: {compras_df.shape[0]}")
compras_df.head()

Total de compras: 101387


,numero_controle_pncp,srp,objeto_compra,codigo_modalidade,nome_modalidade
0,63025530000104-1-000756/2026,False,Aquisição de insumos para controle de vetores.,8,Dispensa
1,04384829000196-1-000151/2026,False,AQUISIÇÃO DE INSUMOS DE LABORATÓRIO,8,Dispensa
2,95440517000108-1-000065/2026,False,Aquisição de seringa descartável graduada em u...,8,Dispensa
3,16695025000197-1-000055/2026,False,"Aquisição de materiais de limpeza, visando ate...",8,Dispensa
4,44763928000101-1-000095/2026,False,AQUISCAO DE MATERIAL ODONTOLOGICO,8,Dispensa


- verifica se há linhas com `objeto_compra` em branco

In [15]:
objeto_compra_nulo = compras_df[compras_df['objeto_compra'].isnull()]
objeto_compra_nulo_count = objeto_compra_nulo.shape[0]

objeto_compra_nulo.head()

,numero_controle_pncp,srp,objeto_compra,codigo_modalidade,nome_modalidade
50036,11020634000122-1-000048/2024,False,NaN,8,Dispensa


## Definição de regras e funções

In [16]:
#----------------------------------
# Objetivo - Buscar termos relacionados a compras judiciais (ex: "judicial" e "justiça")
#----------------------------------

# \b: limita a busca a palavras completas.
# judic\w*: encontra palavras iniciadas por "judic", como "judicial" e "judiciais".
# justica: encontra "justiça" após a remoção de acentos realizada por limpa_texto().
# REGEX_TERMO_JUDICIAL = r"\b(?:judic\w*|justica)\b"

REGEX_TERMO_JUDICIAL = r"judic"

def possui_indicativo_judicial(descricao):
    """Verifica se uma descrição contém indicativo de compra judicial."""
    if pd.isna(descricao):
        return False
    
    if pd.isnull(descricao):
        return False

    texto_descricao = limpa_texto(str(descricao))
    return bool(re.search(REGEX_TERMO_JUDICIAL, texto_descricao))

def limpa_texto(texto):
    """
    Limpa o texto fornecido realizando uma série de transformações:

    1. Converte o texto para minúsculas.
    2. Remove stopwords em português.
    3. Remove acentos.
    4. Remove espaços em branco adicionais.

    Parâmetros:
    -----------
    texto : str
        O texto que será processado e limpo.

    Retorno:
    --------
    str
        O texto limpo após todas as transformações.

    Exemplo:
    --------
    >>> limpa_texto("Aminofilina 24mg/mL - Solução injetável - Ampola com 10mLBR0292402")
    'aminofilina 24mg/ml - solucao injetavel - ampola 10mlbr0292402'
    """
    # Deixa o texto em minúsculo
    texto = texto.lower()

    # Lista de stopwords em português
    stop_words = set(stopwords.words('portuguese'))

    # Remove stop words
    palavras = texto.split()
    palavras_filtradas = [palavra for palavra in palavras if palavra.lower() not in stop_words]
    texto = " ".join(palavras_filtradas)

    # remove acentos
    texto = ''.join(c for c in unicodedata.normalize('NFKD', texto) if not unicodedata.combining(c))

    # Remove espaços em branco adicionais
    texto = ' '.join(texto.split())

    return(texto)

## Aplica função

In [17]:
compras_df["compra_judicial"] = compras_df["objeto_compra"].apply(possui_indicativo_judicial)
compras_df.head()

,numero_controle_pncp,srp,objeto_compra,codigo_modalidade,nome_modalidade,compra_judicial
0,63025530000104-1-000756/2026,False,Aquisição de insumos para controle de vetores.,8,Dispensa,False
1,04384829000196-1-000151/2026,False,AQUISIÇÃO DE INSUMOS DE LABORATÓRIO,8,Dispensa,False
2,95440517000108-1-000065/2026,False,Aquisição de seringa descartável graduada em u...,8,Dispensa,False
3,16695025000197-1-000055/2026,False,"Aquisição de materiais de limpeza, visando ate...",8,Dispensa,False
4,44763928000101-1-000095/2026,False,AQUISCAO DE MATERIAL ODONTOLOGICO,8,Dispensa,False


## Análise dos resultados

### Primeiro, vamos separar somente o dataset onde indicativo de compra judicial é `FALSE`

In [18]:
compras_nao_judiciais_df = compras_df[~compras_df["compra_judicial"]].copy()
print(f"Total de compras não judiciais: {compras_nao_judiciais_df.shape[0]}")
compras_nao_judiciais_df.head()

Total de compras não judiciais: 85617


,numero_controle_pncp,srp,objeto_compra,codigo_modalidade,nome_modalidade,compra_judicial
0,63025530000104-1-000756/2026,False,Aquisição de insumos para controle de vetores.,8,Dispensa,False
1,04384829000196-1-000151/2026,False,AQUISIÇÃO DE INSUMOS DE LABORATÓRIO,8,Dispensa,False
2,95440517000108-1-000065/2026,False,Aquisição de seringa descartável graduada em u...,8,Dispensa,False
3,16695025000197-1-000055/2026,False,"Aquisição de materiais de limpeza, visando ate...",8,Dispensa,False
4,44763928000101-1-000095/2026,False,AQUISCAO DE MATERIAL ODONTOLOGICO,8,Dispensa,False


- Verificar se dentro desse dataset existem algum indicativo de compra judicial que não foi capturado

In [20]:
compras_nao_judiciais_nao_capturadas = compras_nao_judiciais_df[compras_nao_judiciais_df["objeto_compra"].str.contains("JUDIC")].copy()
compras_nao_judiciais_nao_capturadas.head()

,numero_controle_pncp,srp,objeto_compra,codigo_modalidade,nome_modalidade,compra_judicial


### Filtra linhas onde o indicativo de compras judiciais é `true`

In [ ]:
apenas_compras_judiciais_df = compras_df[compras_df["compra_judicial"]].copy()
print(f"Total de compras judiciais encontradas: {apenas_compras_judiciais_df.shape[0]}")
apenas_compras_judiciais_df.head()

Total de compras judiciais encontradas: 15770


,numero_controle_pncp,srp,objeto_compra,codigo_modalidade,nome_modalidade,compra_judicial
10,47970769000104-1-000136/2026,False,AQUISIÇÃO DE MEDICAMENTOS PARA ATENDIMENTO DE ...,8,Dispensa,True
23,46523239000147-1-000120/2026,False,AQUISIÇÃO DE DUPILUMABE PARA ATENDIMENTO DE DE...,8,Dispensa,True
32,63762025000142-1-000022/2026,False,[LICITANET] - AQUISIÇÃO DE MEDICAMENTOS JUDICI...,8,Dispensa,True
45,18240119000105-1-000051/2026,False,[Portal de Compras Públicas] - Aquisição emerg...,8,Dispensa,True
52,46177531000155-1-000087/2026,False,AQUISICAO DE MEDICAMENTOS PARA ATENDER DETERMI...,8,Dispensa,True


- Exportar para diretório de saída

In [ ]:
SAIDA_SOMENTE_JUDICIAIS = apenas_compras_judiciais_df.to_csv(f"{base_dir}/{OUTPUT_COMPRAS_JUDICIAIS}", index=False)